# 03. ML Agent — 머신러닝 Tool 통합
**Day 14**

> Day 8~11에서 배운 머신러닝 모델을 **LangChain `@tool`** 로 감싸 Agent가 호출합니다.  
> 구현 코드: `ml_tools/` 폴더 (로지스틱 회귀 · XGBoost · One-Class SVM)

---
## 0. 설치 & 경로

In [36]:
!pip install openai python-dotenv langchain langchain-openai langchain-core langchain-classic scikit-learn xgboost joblib

In [37]:
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = __import__('os').getenv('OPENAI_API_KEY')

WORKDIR = Path.cwd()
ML_TOOLS_DIR = WORKDIR / 'ml_tools'
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))

print('WORKDIR:', WORKDIR)
print('ML_TOOLS:', ML_TOOLS_DIR)
for p in sorted((ML_TOOLS_DIR / 'data').glob('*.csv')):
    print(' - data/', p.name)

WORKDIR: /Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/14일차
ML_TOOLS: /Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/14일차/ml_tools
 - data/ anomaly_sensor.csv
 - data/ failure_equipment.csv


---
## 1. Day 8~11 ML 방법론 요약

| Tool | 강의 | 알고리즘 | Agent가 쓰는 상황 |
|------|------|----------|-------------------|
| `predict_failure_logistic` | 로지스틱 회귀 | 설비 고장 Yes/No + **확률** (해석·설명) |
| `predict_failure_xgboost` | XGBoost | 불균형 고장 데이터 **예측 성능** 우선 |
| `detect_anomaly_oneclass` | One-Class SVM | 정상 기준 **이상 탐지** (2D 센서) |

공통 패턴:
1. `ml_tools/`에서 **사전 학습 모델** 로드 (`models/*.joblib`)
2. `@tool` 함수가 **문자열**로 결과 반환 → Agent가 읽고 답변

---
## Step 1. ML Tool 불러오기

In [38]:
from ml_tools import ML_TOOLS, predict_failure_logistic, predict_failure_xgboost, detect_anomaly_oneclass

print('등록된 ML Tool:')
for t in ML_TOOLS:
    print(f' - {t.name}: {t.description[:60]}...')

등록된 ML Tool:
 - predict_failure_logistic: 설비 고장(Failure) 여부를 로지스틱 회귀(Day 2-2)로 예측합니다.

    Args:
     ...
 - predict_failure_xgboost: 설비 고장(Failure) 여부를 XGBoost(Day 3-2)로 예측합니다.

    Args:
     ...
 - detect_anomaly_oneclass: 2차원 센서값이 정상 범위인지 One-Class SVM(Day 4-3 / SVDD)으로 탐지합니다.

   ...


---
## ✏️ 실습 1. `@tool` 직접 만들기

**목표:** 현재 시각을 `HH:MM` 형식으로 반환하는 `get_time_hhmm` Tool을 만듭니다.

**힌트:** `from langchain_core.tools import tool` · `datetime.now().strftime('%H:%M')`

In [39]:

from datetime import datetime
from langchain_core.tools import tool

@tool
def get_time_hhmm() -> str:
    """현재 시각을 HH:MM 형식으로 반환합니다."""
    return datetime.now().strftime('%H:%M')

print(get_time_hhmm.invoke({}))

17:11


---
## Step 2. Tool 단독 테스트

Agent 없이 `.invoke()`로 먼저 확인합니다. (14.1의 `get_current_time.invoke`와 동일)

In [40]:
# 설비 고장 예측 — 로지스틱 회귀 (Day 2-2)
print(predict_failure_logistic.invoke({
    'temperature': 67,
    'humidity': 82,
    'operator': 'Operator1',
    'hours_since_previous_failure': 90,
}))

[로지스틱 회귀] 예측: No
- 고장(Failure) 확률: 1.98%
- 정상(No) 확률: 98.02%


In [41]:
# 설비 고장 예측 — XGBoost (Day 3-2)
print(predict_failure_xgboost.invoke({
    'temperature': 67,
    'humidity': 82,
    'operator': 'Operator1',
    'hours_since_previous_failure': 90,
}))

[XGBoost] 예측: No
- 고장(Failure) 확률: 0.74%
- 정상(No) 확률: 99.26%
- 주요 변수(학습 기준): Temperature(0.217), Hours Since Previous Failure(0.138), Humidity(0.094)


In [42]:
# 이상 탐지 — One-Class SVM (Day 4-3 / SVDD)
samples = [(1.0, 1.5, '정상에 가까운 값'), (5.0, -4.0, '이상에 가까운 값')]
for f0, f1, label in samples:
    print(f'[{label}]')
    print(detect_anomaly_oneclass.invoke({'feature_0': f0, 'feature_1': f1}))
    print('-' * 40)

[정상에 가까운 값]
[One-Class SVM] 판정: 이상
- decision_score: -0.0711
- raw_prediction: -1 (+1 정상 / -1 이상)
----------------------------------------
[이상에 가까운 값]
[One-Class SVM] 판정: 이상
- decision_score: -0.9268
- raw_prediction: -1 (+1 정상 / -1 이상)
----------------------------------------


---
## 센서값 직접 바꿔 `.invoke()`

`detect_anomaly_oneclass`에 임의의 `feature_0`, `feature_1` 값을 넣어 결과를 확인하세요.

**관찰:** decision_score가 0에 가까울수록 경계선 샘플입니다.

In [43]:
# ── 여기에 작성 ──
my_f0 = 1000
my_f1 = 100000
print(detect_anomaly_oneclass.invoke({'feature_0': my_f0, 'feature_1': my_f1}))

[One-Class SVM] 판정: 이상
- decision_score: -0.9270
- raw_prediction: -1 (+1 정상 / -1 이상)


---
## Step 3. AgentExecutor — ML Tool 통합

14.1과 같이 `create_tool_calling_agent` + `AgentExecutor`를 씁니다.

In [44]:
from langchain_openai import ChatOpenAI
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.1)

agent_tools = ML_TOOLS
print([t.name for t in agent_tools])

prompt = ChatPromptTemplate.from_messages([
    ('system', '''당신은 제조 설비 분석을 돕는 AI입니다. 질문에 맞는 ML 도구를 골라 사용하세요.

- 설비 고장(Failure) 확률·해석 → predict_failure_logistic
- 설비 고장 예측 (XGBoost, 불균형 데이터) → predict_failure_xgboost
- 2D 센서값 정상/이상 판정 → detect_anomaly_oneclass

온도·습도·작업자·이전 고장 경과시간이 주어지면 ML 예측 도구를 사용하세요.
센서 feature_0, feature_1 값이 주어지면 이상 탐지 도구를 사용하세요.
도구 결과에 없으면 추측하지 마세요. 한국어로 답하세요.'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

['predict_failure_logistic', 'predict_failure_xgboost', 'detect_anomaly_oneclass']


---
## ✏️ 실습 3. 시스템 프롬프트 수정

Step 3의 `prompt` **system** 문자열에 아래 규칙을 **한 줄** 추가한 뒤, Agent를 다시 만들어 보세요.

> *"로지스틱과 XGBoost 결과를 비교해 달라고 하면 두 Tool을 모두 호출하세요."*

**힌트:** `agent` · `executor` 변수를 다시 할당하면 됩니다.

In [45]:
# ── 여기에 작성 ──
# (Step 3 prompt 복사 후 system 문자열에 규칙 1줄 추가)


prompt = ChatPromptTemplate.from_messages([
    ('system', '''당신은 제조 설비 분석을 돕는 AI입니다. 질문에 맞는 ML 도구를 골라 사용하세요.

- 설비 고장(Failure) 확률·해석 → predict_failure_logistic
- 설비 고장 예측 (XGBoost, 불균형 데이터) → predict_failure_xgboost
- 2D 센서값 정상/이상 판정 → detect_anomaly_oneclass

온도·습도·작업자·이전 고장 경과시간이 주어지면 ML 예측 도구를 사용하세요.
센서 feature_0, feature_1 값이 주어지면 이상 탐지 도구를 사용하세요.

로지스틱과 XGBoost 결과를 비교해 달라고 하면 두 Tool을 모두 호출하세요.

도구 결과에 없으면 추측하지 마세요. 한국어로 답하세요.'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

In [46]:
# 로지스틱 회귀 Tool 선택 유도
r = executor.invoke({
    'input': '온도 67, 습도 82, Operator1, 이전 고장 후 90시간 — 로지스틱이랑 XGBoost으로 고장 확률 예측하고 결과를 서로 비교해줘. ',
    'chat_history': [],
})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `predict_failure_logistic` with `{'temperature': 67, 'humidity': 82, 'operator': 'Operator1', 'hours_since_previous_failure': 90}`


[로지스틱 회귀] 예측: No
- 고장(Failure) 확률: 1.98%
- 정상(No) 확률: 98.02%
Invoking: `predict_failure_xgboost` with `{'temperature': 67, 'humidity': 82, 'operator': 'Operator1', 'hours_since_previous_failure': 90}`


[XGBoost] 예측: No
- 고장(Failure) 확률: 0.74%
- 정상(No) 확률: 99.26%
- 주요 변수(학습 기준): Temperature(0.217), Hours Since Previous Failure(0.138), Humidity(0.094)로지스틱 회귀와 XGBoost를 사용한 설비 고장 확률 예측 결과는 다음과 같습니다.

### 로지스틱 회귀
- 예측: **No** (고장 없음)
- 고장(Failure) 확률: **1.98%**
- 정상(No) 확률: **98.02%**

### XGBoost
- 예측: **No** (고장 없음)
- 고장(Failure) 확률: **0.74%**
- 정상(No) 확률: **99.26%**
- 주요 변수(학습 기준):
  - Temperature: **0.217**
  - Hours Since Previous Failure: **0.138**
  - Humidity: **0.094**

### 결과 비교
- 두 모델 모두 고장 없음(No)으로 예측하였으며, 고장 확률은 로지스틱 회귀가 1.98%로 더 높고, XGBoost는 0.74%로 더 낮습니다. 
- XGBoost는 주요 변수의 중요도를 제공하여, 온도와 이전 고장

---
## Step 4. Agent 데모 질문

In [47]:
# 로지스틱 회귀 Tool 선택 유도
r = executor.invoke({
    'input': '온도 67, 습도 82, Operator1, 이전 고장 후 90시간 — 로지스틱으로 고장 확률 알려줘',
    'chat_history': [],
})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `predict_failure_logistic` with `{'temperature': 67, 'humidity': 82, 'operator': 'Operator1', 'hours_since_previous_failure': 90}`


[로지스틱 회귀] 예측: No
- 고장(Failure) 확률: 1.98%
- 정상(No) 확률: 98.02%로지스틱 회귀 예측 결과는 다음과 같습니다:

- 고장(Failure) 확률: 1.98%
- 정상(No) 확률: 98.02%

따라서, 고장이 발생할 가능성은 매우 낮습니다.

> Finished chain.
로지스틱 회귀 예측 결과는 다음과 같습니다:

- 고장(Failure) 확률: 1.98%
- 정상(No) 확률: 98.02%

따라서, 고장이 발생할 가능성은 매우 낮습니다.


In [48]:
# XGBoost Tool 선택 유도
r = executor.invoke({
    'input': '같은 조건으로 XGBoost 모델 예측도 해줘',
    'chat_history': [],
})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `predict_failure_xgboost` with `{'temperature': 67, 'humidity': 82, 'operator': 'Operator1', 'hours_since_previous_failure': 5}`


[XGBoost] 예측: No
- 고장(Failure) 확률: 0.86%
- 정상(No) 확률: 99.14%
- 주요 변수(학습 기준): Temperature(0.217), Hours Since Previous Failure(0.138), Humidity(0.094)XGBoost 모델 예측 결과는 다음과 같습니다:

- 고장(Failure) 예측: **No**
- 고장 확률: **0.86%**
- 정상(No) 확률: **99.14%**
- 주요 변수(학습 기준):
  - 온도(Temperature): 0.217
  - 이전 고장 경과 시간(Hours Since Previous Failure): 0.138
  - 습도(Humidity): 0.094

이 결과를 바탕으로 추가적인 질문이 있으시면 말씀해 주세요.

> Finished chain.
XGBoost 모델 예측 결과는 다음과 같습니다:

- 고장(Failure) 예측: **No**
- 고장 확률: **0.86%**
- 정상(No) 확률: **99.14%**
- 주요 변수(학습 기준):
  - 온도(Temperature): 0.217
  - 이전 고장 경과 시간(Hours Since Previous Failure): 0.138
  - 습도(Humidity): 0.094

이 결과를 바탕으로 추가적인 질문이 있으시면 말씀해 주세요.


In [49]:
# One-Class SVM 이상 탐지
r = executor.invoke({
    'input': '센서 feature_0=1.0, feature_1=1.5 일 때 정상인지 이상인지 판정해줘',
    'chat_history': [],
})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `detect_anomaly_oneclass` with `{'feature_0': 1.0, 'feature_1': 1.5}`


[One-Class SVM] 판정: 이상
- decision_score: -0.0711
- raw_prediction: -1 (+1 정상 / -1 이상)센서 feature_0=1.0, feature_1=1.5일 때 판정 결과는 "이상"입니다. (decision_score: -0.0711)

> Finished chain.
센서 feature_0=1.0, feature_1=1.5일 때 판정 결과는 "이상"입니다. (decision_score: -0.0711)


---
## ✏️ 실습 4. Agent에 본인 질문 작성

**고장 예측** 또는 **이상 탐지** 중 하나를 골라, Step 4 데모와 다른 조건으로 `executor.invoke`를 직접 호출하세요.

**예시:** *"온도 80, 습도 90, Operator3, 이전 고장 10시간 — XGBoost로 예측해줘"*

In [50]:
# One-Class SVM 이상 탐지
r = executor.invoke({
    'input': '온도 80, 습고 90, Operator3, 이전 고장 시간 10시간 일때 XGBoost로 고장여부 예측해줘. ',
    'chat_history': [],
})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `predict_failure_xgboost` with `{'temperature': 80, 'humidity': 90, 'operator': 'Operator3', 'hours_since_previous_failure': 10}`


[XGBoost] 예측: No
- 고장(Failure) 확률: 5.75%
- 정상(No) 확률: 94.25%
- 주요 변수(학습 기준): Temperature(0.217), Hours Since Previous Failure(0.138), Humidity(0.094)XGBoost 예측 결과는 다음과 같습니다:

- 고장(Failure) 여부: **No**
- 고장 확률: **5.75%**
- 정상 확률: **94.25%**

주요 변수는 다음과 같습니다:
- 온도(Temperature): 0.217
- 이전 고장 경과 시간(Hours Since Previous Failure): 0.138
- 습도(Humidity): 0.094

> Finished chain.
XGBoost 예측 결과는 다음과 같습니다:

- 고장(Failure) 여부: **No**
- 고장 확률: **5.75%**
- 정상 확률: **94.25%**

주요 변수는 다음과 같습니다:
- 온도(Temperature): 0.217
- 이전 고장 경과 시간(Hours Since Previous Failure): 0.138
- 습도(Humidity): 0.094


---
## Step 5. 멀티턴 — `chat_history`

In [51]:
from langchain_core.messages import HumanMessage, AIMessage

class ChatSession:
    def __init__(self, executor):
        self.executor = executor
        self.history = []

    def ask(self, question: str) -> str:
        result = self.executor.invoke({'input': question, 'chat_history': self.history})
        answer = result['output']
        self.history.extend([HumanMessage(content=question), AIMessage(content=answer)])
        return answer

session = ChatSession(executor)
print('1:', session.ask('온도 70, 습도 75, Operator2, 이전 고장 120시간 — 고장 날까?'))
print('2:', session.ask('방금 답에서 고장 확률만 숫자로 다시 말해줘'))



> Entering new AgentExecutor chain...

Invoking: `predict_failure_logistic` with `{'temperature': 70, 'humidity': 75, 'operator': 'Operator2', 'hours_since_previous_failure': 120}`


[로지스틱 회귀] 예측: No
- 고장(Failure) 확률: 11.09%
- 정상(No) 확률: 88.91%
Invoking: `predict_failure_xgboost` with `{'temperature': 70, 'humidity': 75, 'operator': 'Operator2', 'hours_since_previous_failure': 120}`


[XGBoost] 예측: Yes
- 고장(Failure) 확률: 96.66%
- 정상(No) 확률: 3.34%
- 주요 변수(학습 기준): Temperature(0.217), Hours Since Previous Failure(0.138), Humidity(0.094)로지스틱 회귀 모델의 예측 결과는 다음과 같습니다:
- 고장(Failure) 예측: **아니오 (No)**
- 고장 확률: **11.09%**
- 정상 확률: **88.91%**

XGBoost 모델의 예측 결과는 다음과 같습니다:
- 고장(Failure) 예측: **예 (Yes)**
- 고장 확률: **96.66%**
- 정상 확률: **3.34%**
- 주요 변수: 온도(Temperature), 이전 고장 경과 시간(Hours Since Previous Failure), 습도(Humidity)

두 모델의 예측 결과가 상이하므로, 상황에 따라 추가적인 분석이 필요할 수 있습니다.

> Finished chain.
1: 로지스틱 회귀 모델의 예측 결과는 다음과 같습니다:
- 고장(Failure) 예측: **아니오 (No)**
- 고장 확률: **11.09%**
- 정상 확률: **88.91%**

XGBoost

---
## ✏️ 실습 5. 멀티턴 후속 질문

`ChatSession`으로 1턴 예측 후, **2턴에서 맥락을 이어가는** 후속 질문을 직접 작성하세요.

**예시:** 1턴 고장 예측 → 2턴 *"그럼 같은 조건으로 이상 탐지도 해줘 feature_0=2, feature_1=2"*

In [52]:
# ── 여기에 작성 ──
# session2 = ChatSession(executor)
# q1 = '...'
# q2 = '...'  # 1턴 답을 참조하는 후속 질문
# print('1:', session2.ask(q1))
# print('2:', session2.ask(q2))

session3 = ChatSession(executor)
print('1:', session3.ask('온도 40, 습도 85, Operator2, 이전 고장 30시간 — 고장 날까? 로지스틱 회귀로 예측해봐'))
print('2:', session3.ask('방금 조건에서 온도만 80으로 올리면 고장 여부가 어떻게 바뀌니?'))



> Entering new AgentExecutor chain...

Invoking: `predict_failure_logistic` with `{'temperature': 40, 'humidity': 85, 'operator': 'Operator2', 'hours_since_previous_failure': 30}`


[로지스틱 회귀] 예측: No
- 고장(Failure) 확률: 0.00%
- 정상(No) 확률: 100.00%로지스틱 회귀 예측 결과, 고장(Failure) 확률은 0.00%로, 정상(No) 확률은 100.00%입니다. 따라서 고장이 날 가능성은 없습니다.

> Finished chain.
1: 로지스틱 회귀 예측 결과, 고장(Failure) 확률은 0.00%로, 정상(No) 확률은 100.00%입니다. 따라서 고장이 날 가능성은 없습니다.


> Entering new AgentExecutor chain...

Invoking: `predict_failure_logistic` with `{'temperature': 80, 'humidity': 85, 'operator': 'Operator2', 'hours_since_previous_failure': 30}`


[로지스틱 회귀] 예측: Yes
- 고장(Failure) 확률: 63.76%
- 정상(No) 확률: 36.24%온도를 80으로 올리면 로지스틱 회귀 예측 결과, 고장(Failure) 확률은 63.76%로, 정상(No) 확률은 36.24%입니다. 따라서 고장이 날 가능성이 있습니다.

> Finished chain.
2: 온도를 80으로 올리면 로지스틱 회귀 예측 결과, 고장(Failure) 확률은 63.76%로, 정상(No) 확률은 36.24%입니다. 따라서 고장이 날 가능성이 있습니다.


---
## Step 6. (선택) 14.1 Agent와 합치기

RAG · 웹 검색 · ML을 한 Agent에 넣으려면 `agent_tools` 리스트에 합치면 됩니다.

### ✏️ 실습 6

1. **14.1 노트북에 정의**된 `get_current_time`, `search_pdf_tool`, `web_search`코드를 복사해 선언합니다.
2. 아래 셀에서 `agent_tools`를 채우고, `system` 프롬프트에 RAG·웹·ML Tool 용도를 각각 한 줄씩 추가하세요.
3. (선택) 합친 Agent로 *"석사 수업연한이 몇 학기야?"* 와 *"온도 70 습도 80 고장 예측"* 을 연속 질문해 보세요.

In [53]:
from datetime import datetime
import os
import pytz
import yfinance as yf
from langchain_core.tools import tool, create_retriever_tool
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

@tool
def get_current_time(timezone: str, location: str) -> str:
    """현재 시각. timezone 예: Asia/Seoul, location 예: 서울"""
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime('%Y-%m-%d %H:%M:%S')
    return f'{timezone} ({location}) 현재시각 {now}'

@tool
def get_yf_stock_history(ticker: str, period: str) -> str:
    """주식 가격 조회. ticker 예: TSLA, period 예: 1mo"""
    history = yf.Ticker(ticker).history(period=period)
    return history.tail(3).to_string() if not history.empty else '데이터 없음'

web_search = DuckDuckGoSearchResults(num_results=3)

# RAG — search_pdf_tool (14.1과 동일)
RAG_SAMPLES = os.path.join(os.getcwd(), 'samples', 'pdf_samples')
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
loader = PyPDFLoader(os.path.join(RAG_SAMPLES, '학칙.pdf'))
all_splits = text_splitter.split_documents(loader.load())
loader_news = PyPDFLoader(os.path.join(RAG_SAMPLES, '보도자료.pdf'))
all_splits.extend(text_splitter.split_documents(loader_news.load()))

embedding = OpenAIEmbeddings(model='text-embedding-3-large', api_key=OPENAI_API_KEY)
persist_directory = 'chroma_store'
if not os.path.exists(persist_directory):
    vectorstore = Chroma.from_documents(
        documents=all_splits, embedding=embedding, persist_directory=persist_directory
    )
else:
    vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embedding)

search_pdf_tool = create_retriever_tool(
    vectorstore.as_retriever(search_kwargs={'k': 3}),
    name='search_pdf_documents',
    description='pdf samples 안의 문서함에서 질문과 관련된 내용을 검색합니다.',
)

In [54]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

RAG_SAMPLES = os.path.join(os.getcwd(), 'samples', 'pdf_samples')
print('RAG_SAMPLES:', RAG_SAMPLES)
for p in sorted(Path(RAG_SAMPLES).glob('*.pdf')):
    print(' -', p.name)

RAG_SAMPLES: /Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/14일차/samples/pdf_samples
 - A survey on large language model based autonomous agents.pdf
 - NeurIPS-2023-analyzing-vision-transformers-for-image-classification-in-class-embedding-space-Paper-Conference.pdf
 - Physics-Guided Deep Learning.pdf
 - data2vec.pdf
 - supervised-contrastive-learning-Paper.pdf
 - 보도자료.pdf
 - 학칙.pdf


In [55]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
import os

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

# 학칙.pdf
loader = PyPDFLoader(os.path.join(RAG_SAMPLES, '학칙.pdf'))
all_splits = text_splitter.split_documents(loader.load())

# news.pdf
loader_news = PyPDFLoader(os.path.join(RAG_SAMPLES, '보도자료.pdf'))
news_splits = text_splitter.split_documents(loader_news.load())
all_splits.extend(news_splits)

embedding = OpenAIEmbeddings(
    model='text-embedding-3-large', api_key=OPENAI_API_KEY
)

persist_directory = 'chroma_store'	

# 저장된 크로마 DB가 없다면 새로 만들기
if not os.path.exists(persist_directory):
    print("Creating new Chroma store")
    vectorstore = Chroma.from_documents(
        documents=all_splits,
        embedding=embedding,
        persist_directory=persist_directory
    )

else:
    print("Loading existing Chroma store")
    vectorstore = Chroma(		
        persist_directory=persist_directory, 
        embedding_function=embedding
    )
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

Loading existing Chroma store


In [56]:
from langchain_core.tools import create_retriever_tool

search_pdf_tool = create_retriever_tool(
    retriever,
    name='search_pdf_documents',
    description='pdf samples 안의 문서함에서 질문과 관련된 내용을 검색합니다.',
)

In [57]:
# ── 여기에 작성 (14.1 실행 후) ──
agent_tools = [
    get_current_time,
    search_pdf_tool,
    web_search,
    *ML_TOOLS,
]

combined_prompt = ChatPromptTemplate.from_messages([
    ('system', '''당신은 제조·학칙·웹·ML 분석을 돕는 AI입니다. 질문에 맞는 도구를 골라 사용하세요.

- samples/pdf_samples 문서(학칙, 논문, 보도자료 등) 검색 → search_pdf_documents
- 최신 뉴스·시사·웹에서 찾을 수 있는 정보 → duckduckgo_results_json
- 현재 시각(도시·타임존) → get_current_time
- 설비 고장(Failure) 확률·해석이 필요할 때 → predict_failure_logistic
- 설비 고장 예측(불균형 데이터, XGBoost) → predict_failure_xgboost
- 2D 센서값(feature_0, feature_1) 정상/이상 판정 → detect_anomaly_oneclass

"지금 몇시" 같은 시각 질문은 get_current_time을 사용하세요 (기본: timezone=Asia/Seoul, location=서울).
온도·습도·작업자·이전 고장 경과시간이 주어지면 ML 예측 도구를 사용하세요.
센서 feature_0, feature_1 값이 주어지면 이상 탐지 도구를 사용하세요.
학칙·수업연한·논문 내용은 search_pdf_documents, 실시간 웹 정보는 duckduckgo_results_json을 사용하세요.
도구 결과에 없으면 추측하지 마세요. 한국어로 답하세요.'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

combined_agent = create_tool_calling_agent(llm, agent_tools, combined_prompt)
combined_executor = AgentExecutor(agent=combined_agent, tools=agent_tools, verbose=True, max_iterations=8)

session = ChatSession(combined_executor)  # 대화 루프는 합친 Agent 사용
print('등록된 Tool:', [t.name for t in agent_tools])

등록된 Tool: ['get_current_time', 'search_pdf_documents', 'duckduckgo_results_json', 'predict_failure_logistic', 'predict_failure_xgboost', 'detect_anomaly_oneclass']


---
## Step 7. `input()` 대화 루프

In [59]:
print('종료: exit')
while True:
    user_input = input('사용자: ').strip()
    if user_input.lower() in ('exit', 'quit', '종료'):
        break
    if not user_input:
        continue
    print('AI:', session.ask(user_input))

종료: exit


> Entering new AgentExecutor chain...

Invoking: `predict_failure_logistic` with `{'temperature': 40, 'humidity': 85, 'operator': 'Operator2', 'hours_since_previous_failure': 30}`


[로지스틱 회귀] 예측: No
- 고장(Failure) 확률: 0.00%
- 정상(No) 확률: 100.00%
Invoking: `predict_failure_xgboost` with `{'temperature': 40, 'humidity': 85, 'operator': 'Operator2', 'hours_since_previous_failure': 30}`


[XGBoost] 예측: No
- 고장(Failure) 확률: 0.18%
- 정상(No) 확률: 99.82%
- 주요 변수(학습 기준): Temperature(0.217), Hours Since Previous Failure(0.138), Humidity(0.094)예측 결과는 다음과 같습니다:

- **로지스틱 회귀**:
  - 예측: **No** (고장 없음)
  - 고장(Failure) 확률: **0.00%**
  - 정상(No) 확률: **100.00%**

- **XGBoost**:
  - 예측: **No** (고장 없음)
  - 고장(Failure) 확률: **0.18%**
  - 정상(No) 확률: **99.82%**
  - 주요 변수(학습 기준):
    - Temperature: 0.217
    - Hours Since Previous Failure: 0.138
    - Humidity: 0.094

결과적으로 두 모델 모두 고장이 발생하지 않을 것으로 예측하고 있습니다.

> Finished chain.
AI: 예측 결과는 다음과 같습니다:

- **로지스틱 회귀**:
  - 예측: **No** (고장 없음)
  - 고장(Failure) 확률

---
## 정리

| 파일 | 역할 |
|------|------|
| `ml_tools/logistic_regression_tool.py` | Day 9 로지스틱 회귀 |
| `ml_tools/xgboost_classifier_tool.py` | Day 10 XGBoost |
| `ml_tools/oneclass_svm_tool.py` | Day 11 One-Class SVM |
| `ml_tools/models/*.joblib` | 사전 학습 모델 (자동 생성) |

> ML Tool은 **추론만** 수행합니다. 재학습이 필요하면 각 `.py`를 `python logistic_regression_tool.py`처럼 직접 실행하세요.